## Preprocessing of Mkechinov E-commerce data: from event logs to session sequences

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../../data/mkechinov/2019-Nov.csv')
print(df.shape)
df.head()

(67501979, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [3]:

# CLEAN DATA

# Convert event_time into datetime
df["event_time"] = pd.to_datetime(
    df["event_time"],
    errors="coerce",
    utc=True
)

# Standardize event names
df["event_type"] = (
    df["event_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Standardize category and brand information
df["category_code"] = (
    df["category_code"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["brand"] = (
    df["brand"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Remove rows without essential information
df = df.dropna(
    subset=["user_id", "user_session", "event_time", "event_type"]
)

# Remove exact duplicate records
before_duplicates = len(df)

df = df.drop_duplicates()

duplicates_removed = before_duplicates - len(df)

# Sort events chronologically within each session
df = df.sort_values(
    ["user_id", "user_session", "event_time"],
    kind="mergesort"
).reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Duplicates removed:", duplicates_removed)

print("\nEvent distribution:")
print(df["event_type"].value_counts())

print("\nMissing values:")
print(df.isna().sum())


Cleaned shape: (67401450, 9)
Duplicates removed: 100519

Event distribution:
event_type
view        63554512
cart         2930008
purchase      916930
Name: count, dtype: Int64

Missing values:
event_time              0
event_type              0
product_id              0
category_id             0
category_code    21871420
brand             9215007
price                   0
user_id                 0
user_session            0
dtype: int64


### Validation 

In [5]:

# SESSION VALIDATION

print("\n========== SESSION VALIDATION ==========")

# Check whether session IDs are associated with multiple users
users_per_session = (
    df.groupby("user_session")["user_id"]
    .nunique()
)

print(
    "Session IDs associated with multiple users:",
    (users_per_session > 1).sum()
)

# Session statistics
session_stats = (
    df.groupby(["user_id", "user_session"])
    .agg(
        start=("event_time", "min"),
        end=("event_time", "max"),
        n_events=("event_type", "size"),
        n_products=("product_id", "nunique")
    )
)

session_stats["duration_min"] = (
    session_stats["end"] - session_stats["start"]
).dt.total_seconds() / 60

print("\nSession duration statistics:")
print(session_stats["duration_min"].describe())

print("\nSession length statistics:")
print(session_stats["n_events"].describe())



========== SESSION VALIDATION ==========
Session IDs associated with multiple users: 591

Session duration statistics:
count    1.377664e+07
mean     1.551812e+01
std      3.415173e+02
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+00
75%      4.750000e+00
max      4.230373e+04
Name: duration_min, dtype: float64

Session length statistics:
count    1.377664e+07
mean     4.892444e+00
std      7.857993e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      5.000000e+00
max      4.128000e+03
Name: n_events, dtype: float64


In [6]:

# BUILD SESSION SEQUENCES

# Identify purchase events
df["is_purchase"] = df["event_type"].eq("purchase")

# Create a category representation
# Use category_id when category_code is unavailable
df["category_token"] = (
    df["category_code"]
    .fillna(df["category_id"].astype("string"))
    .fillna("unknown")
)

# Create enriched event tokens
df["event_category"] = (
    df["event_type"] + "_" + df["category_token"]
)

# Build sessions
sessions = (
    df.groupby(["user_id", "user_session"], sort=False)
    .agg(
        events=("event_type", list),
        enriched_events=("event_category", list),
        products=("product_id", list),
        brands=("brand", list),
        has_purchase=("is_purchase", "any"),
        purchase_value=(
            "price",
            lambda x: x[df.loc[x.index, "is_purchase"]].sum()
        ),
        n_unique_products=("product_id", "nunique"),
        start=("event_time", "min"),
        end=("event_time", "max")
    )
    .reset_index()
)

# Number of events
sessions["n_events"] = sessions["events"].str.len()

# Session duration
sessions["duration_min"] = (
    sessions["end"] - sessions["start"]
).dt.total_seconds() / 60

# Session outcome
sessions["outcome"] = np.where(
    sessions["has_purchase"],
    "purchase",
    "no_purchase"
)

# Remove temporary column
sessions = sessions.drop(columns=["has_purchase"])

# Arrange columns
sessions = sessions[
    [
        "user_id",
        "user_session",
        "events",
        "enriched_events",
        "products",
        "brands",
        "outcome",
        "purchase_value",
        "n_events",
        "n_unique_products",
        "duration_min"
    ]
]

In [7]:

# VALIDATION

print("\n========== SESSION SUMMARY ==========")

print("Total sessions:", len(sessions))
print("Unique users:", sessions["user_id"].nunique())

print("\nSession length statistics:")
print(sessions["n_events"].describe())

print("\nSession duration statistics:")
print(sessions["duration_min"].describe())

print("\nUnique products per session:")
print(sessions["n_unique_products"].describe())

print("\nSession outcome distribution:")
print(sessions["outcome"].value_counts())

print("\nEvent distribution:")
print(df["event_type"].value_counts())

print("\nSession length distribution:")
print(sessions["n_events"].value_counts().sort_index())

print("\nFinal dataset:")
display(sessions.head())



========== SESSION SUMMARY ==========
Total sessions: 13776641
Unique users: 3696117

Session length statistics:
count    1.377664e+07
mean     4.892444e+00
std      7.857993e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      5.000000e+00
max      4.128000e+03
Name: n_events, dtype: float64

Session duration statistics:
count    1.377664e+07
mean     1.551812e+01
std      3.415173e+02
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+00
75%      4.750000e+00
max      4.230373e+04
Name: duration_min, dtype: float64

Unique products per session:
count    1.377664e+07
mean     3.046113e+00
std      4.734456e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      3.000000e+00
max      4.128000e+03
Name: n_unique_products, dtype: float64

Session outcome distribution:
outcome
no_purchase    13003427
purchase         773214
Name: count, dtype: int64

Event distribution:
event_type
view        63554512
cart         2930008
purch

,user_id,user_session,events,enriched_events,products,brands,outcome,purchase_value,n_events,n_unique_products,duration_min
0,10300217,d1fdcbf1-bb1f-434b-8f1a-4b77f29a84a0,[view],[view_2053013563424899933],[26300219],[sokolov],no_purchase,0.0,1,1,0.000000
1,29515875,1552ef43-4001-4ab7-a9c7-44e99f1730a9,"[view, view]","[view_furniture.bedroom.bed, view_furniture.be...","[13200917, 13200917]","[<NA>, <NA>]",no_purchase,0.0,2,1,0.083333
2,29515875,27452feb-1b63-41cc-ac67-030d56f6d448,"[view, view, view, view]","[view_electronics.video.tv, view_electronics.v...","[1802034, 1801638, 1801638, 1801638]","[kivi, harper, harper, harper]",no_purchase,0.0,4,2,1.766667
3,29515875,447d6309-4efe-4662-a64d-b19992fbfe1c,[view],[view_electronics.video.tv],[1802034],[kivi],no_purchase,0.0,1,1,0.000000
4,29515875,5dd49822-9349-42d6-a165-cd4642ee606b,[view],[view_furniture.bedroom.bed],[13200917],[<NA>],no_purchase,0.0,1,1,0.000000


In [8]:
from pathlib import Path

output_dir = Path("../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

sessions.to_parquet(
    output_dir / "mkechinov_sessions.parquet",
    index=False
)

print("Sessions exported successfully!")

Sessions exported successfully!
